# Language Model Training with Mamba

> Train a character-level language model using Mamba

This notebook demonstrates training a complete Mamba language model on text data.

- **Level:** Intermediate
- **Time:** ~30 minutes
- **GPU:** Recommended for training

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Parallel Scan Functions

In [ ]:
# Parallel scan implementation
def ssm_parallel_scan(A_bar, B_bar):
    """Parallel scan for SSM with diagonal A"""
    batch, seq_len, d_state = B_bar.shape
    n = 1 << (seq_len - 1).bit_length()
    
    A = torch.zeros(batch, n, d_state, device=A_bar.device, dtype=A_bar.dtype)
    B = torch.zeros(batch, n, d_state, device=B_bar.device, dtype=B_bar.dtype)
    A[:, :seq_len, :] = A_bar
    B[:, :seq_len, :] = B_bar
    
    for stride in range(n.bit_length() - 1):
        step = 1 << stride
        A_even = A[:, ::step*2, :]
        B_even = B[:, ::step*2, :]
        A_odd = A[:, step::step*2, :]
        B_odd = B[:, step::step*2, :]
        
        A[:, step::step*2, :] = A_even * A_odd
        B[:, step::step*2, :] = A_even * B_odd + B_even
    
    return B[:, :seq_len, :]
# export
ssm_parallel_scan = ssm_parallel_scan

## Mamba Block

In [ ]:
class SelectiveSSM(nn.Module):
    """Mamba's Selective State Space Model"""
    
    def __init__(self, d_model, d_state, dt_rank=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.dt_rank = dt_rank
        
        self.A_log = nn.Parameter(torch.randn(d_state))
        self.D = nn.Parameter(torch.ones(d_state))
        self.x_proj = nn.Linear(d_model, dt_rank + d_state * 2, bias=False)
        self.out_proj = nn.Linear(d_state, d_model, bias=False)
    
    def forward(self, x):
        batch, seq_len, d_model = x.shape
        d_state = self.d_state
        
        x_params = self.x_proj(x)
        delta = x_params[:, :, :self.dt_rank]
        B = x_params[:, :, self.dt_rank:self.dt_rank + d_state]
        C = x_params[:, :, self.dt_rank + d_state:]
        
        delta = F.softplus(delta)
        
        A_bar = torch.exp(self.A_log.unsqueeze(0).unsqueeze(0) * delta.unsqueeze(-1))
        
        disc = (A_bar - 1) / torch.where(
            torch.abs(self.A_log.unsqueeze(0).unsqueeze(0)) > 1e-6,
            self.A_log.unsqueeze(0).unsqueeze(0),
            1e-6
        )
        B_bar = disc * B.unsqueeze(-1)
        
        h = ssm_parallel_scan(A_bar, B_bar)
        
        y = torch.einsum('bldn,bln->bld', h, C)
        y = y + x * self.D.unsqueeze(0).unsqueeze(0)
        
        return self.out_proj(y)
# export
SelectiveSSM = SelectiveSSM

In [ ]:
class MambaBlock(nn.Module):
    """Full Mamba Block"""
    
    def __init__(self, d_model, d_state=128, expand=2, dt_rank=16):
        super().__init__()
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        
        self.norm = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2)
        self.ssm = SelectiveSSM(self.d_inner, d_state, dt_rank)
        self.out_proj = nn.Linear(self.d_inner, d_model)
    
    def forward(self, x):
        x_norm = self.norm(x)
        x_inner, gate = self.in_proj(x_norm).chunk(2, dim=-1)
        gate = F.silu(gate)
        ssm_out = self.ssm(gate)
        y = ssm_out * gate
        y = self.out_proj(y)
        return y + x
# export
MambaBlock = MambaBlock

## Language Model

In [ ]:
class MambaLM(nn.Module):
    """Mamba Language Model"""
    
    def __init__(self, vocab_size, d_model=256, n_layers=4, d_state=128):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([MambaBlock(d_model, d_state) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying
        self.lm_head.weight = self.embedding.weight
    
    def forward(self, x, targets=None):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        logits = self.lm_head(x)
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.vocab_size), targets.view(-1))
        
        return logits, loss
    
    @torch.no_grad()
    def generate(self, x, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            x_cond = x[:, -512:] if x.size(1) > 512 else x
            logits, _ = self(x_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            x = torch.cat([x, next_token], dim=1)
        return x
# export
MambaLM = MambaLM

## Data Preparation

In [ ]:
# Sample text data
text = """
The history of natural language processing started in the 1950s. 
In 1950 Alan Turing published Computing Machinery and Intelligence. 
The Georgetown experiment in 1954 involved machine translation. 
Modern large language models are based on transformers introduced in 2017. 
State space models represent an alternative to attention. 
Mamba is a selective state space model that achieves strong performance. 
It combines efficiency with content-based selection. 
The architecture uses parallel scan for efficiency. 
Training Mamba is similar to training other neural networks. 
Text generation is done autoregressively. 
At each step the model predicts the next token. 
The predicted token is fed back as input. 
This process continues until a stop condition is reached. 
Language models learn statistical patterns in text. 
They capture regularities in the training data. 
Larger models can capture more complex patterns. 
They learn syntax semantics and world knowledge. 
Models can generate coherent and fluent text. 
They can be used for translation summarization and question answering.
""".strip()

# Character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {chars[:30]}...")

# Encode/decode
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

encode = lambda s: torch.tensor([char_to_idx[c] for c in s], dtype=torch.long)
decode = lambda idx: ''.join([idx_to_char[i.item()] for i in idx])

# Encode all text
data = encode(text)
print(f"Data size: {len(data)} tokens")

## Training Loop

In [ ]:
def get_batch(data, batch_size, seq_len):
    """Get random batch"""
    ix = torch.randint(len(data) - seq_len, (batch_size,))
    x = torch.stack([data[i:i+seq_len] for i in ix])
    y = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    return x.to(device), y.to(device)

# Training config
batch_size = 32
seq_len = 64
learning_rate = 0.001
max_epochs = 100

# Create model
model = MambaLM(vocab_size=vocab_size, d_model=128, n_layers=2, d_state=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Starting training...")

In [ ]:
train_losses = []

for epoch in range(max_epochs):
    x, y = get_batch(data, batch_size, seq_len)
    
    optimizer.zero_grad()
    logits, loss = model(x, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    train_losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}: Loss = {loss.item():.4f}")

print("\n✅ Training complete!")

## Generate Text

In [ ]:
# Generate text
model.eval()
context = torch.tensor([[char_to_idx['T']]], dtype=torch.long).to(device)

with torch.no_grad():
    generated = model.generate(context, max_new_tokens=200, temperature=0.8)

print("Generated text:")
print("=" * 50)
print(decode(generated[0]))
print("=" * 50)

In [ ]:
# Plot training curve
plt.figure(figsize=(10, 4))
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curve')
plt.grid(True)
plt.show()

## Summary

We built a complete Mamba language model:

1. ✅ Token embeddings with weight tying
2. ✅ Stacked Mamba blocks
3. ✅ Language model head
4. ✅ Text generation

The model learned to generate text based on the training corpus!

---

**Next:** Try training on larger datasets, or compare with Transformer!